# Neura Care Assistant — Phase 4 retraining on context-conditioned data

The first fine-tune learned from short prompts with no context, but the app sends a long RAG prompt (numbered knowledge-base excerpts, the patient's care data, citation and safety rules). On that prompt it gave one-sentence answers, never cited sources, and invented patient facts.

This notebook retrains **Llama-3.2-1B-Instruct** with QLoRA (Unsloth) on `context_dataset.train.jsonl`. Every example in it uses the exact production prompt, so what the model sees in training matches what it sees in the app.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Then run the cells top to bottom (about 20–30 minutes).

You'll need:
- `training/context_dataset.train.jsonl` and `training/context_dataset.val.jsonl` from the repo (cell 4 uploads them)
- a Hugging Face token with **write** access (cell 9)

In [ ]:
# 1. Install dependencies (same setup as the first fine-tune)
!pip install -q unsloth
!pip install -q --upgrade --force-reinstall --no-deps "unsloth[colab-new]"
!pip install -q --upgrade huggingface_hub

In [ ]:
# 2. Load the base model in 4-bit. 1B matches the model the app serves (16 layers, hidden size 2048).
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 3072   # production prompts average ~1,050 tokens; this leaves room for long articles + the answer
BASE_MODEL = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
# 3. LoRA adapter (the only weights that are trained)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 4. Upload BOTH files: training/context_dataset.train.jsonl and training/context_dataset.val.jsonl
from google.colab import files
uploaded = files.upload()
TRAIN_PATH = next(k for k in uploaded if "train" in k)
VAL_PATH = next(k for k in uploaded if "val" in k)
print("train:", TRAIN_PATH, "| val:", VAL_PATH)

In [ ]:
# 5. Apply the chat template and check lengths (nothing may be cut off)
from datasets import load_dataset

train_raw = load_dataset("json", data_files=TRAIN_PATH, split="train")
val_raw = load_dataset("json", data_files=VAL_PATH, split="train")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

train_dataset = train_raw.map(to_text)
val_dataset = val_raw.map(to_text)

lengths = [len(tokenizer(t)["input_ids"]) for t in train_dataset["text"]]
print(f"Train {len(train_dataset)}  Val {len(val_dataset)}  tokens: mean {sum(lengths)/len(lengths):.0f}, max {max(lengths)}")
assert max(lengths) <= MAX_SEQ_LENGTH, "Some examples are longer than MAX_SEQ_LENGTH — raise it"
from collections import Counter
print("kinds:", Counter(train_raw["kind"]))

In [ ]:
# 6. Train on the ANSWERS only: the long prompt is context, not something to imitate.
# Keeps the epoch with the lowest validation loss.
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
stats = trainer.train()
print(stats)
print("Validation loss per epoch:", [round(x["eval_loss"], 4) for x in trainer.state.log_history if "eval_loss" in x])

In [ ]:
# 7. Behaviour check on the validation set (objective, quick): does it cite, abstain and refer to a doctor
# when it should, and avoid numeric dates? Compare these rates with the teacher's (100% by construction).
import re
FastLanguageModel.for_inference(model)

def generate(messages, max_new_tokens=350):
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, temperature=0.4, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

ABSTAIN = re.compile(r"doesn'?t have|does not have|no record|not recorded|isn'?t recorded|not available|don'?t have|do not have|no information", re.I)
DOCTOR = re.compile(r"doctor|physician|GP\b|pharmacist|prescrib|care team|medical professional", re.I)
NUMERIC_DATE = re.compile(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b(?:on|dated|from|since|until|by)\s+\d{1,2}/\d{1,2}\b", re.I)

rows = []
for ex in val_raw:
    system, user = ex["messages"][0]["content"], ex["messages"][1]["content"]
    n_articles = len(re.findall(r"^\[\d+\] ", system, re.M))
    answer = generate([{"role": "system", "content": system}, {"role": "user", "content": user}])
    cites = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    rows.append({
        "kind": ex["kind"],
        "cites_when_should": (bool(cites) if ex["kind"] == "knowledge" and n_articles else None),
        "valid_citations": all(1 <= c <= n_articles for c in cites),
        "abstains": (bool(ABSTAIN.search(answer)) if ex["kind"] == "insufficient" else None),
        "refers_to_doctor": (bool(DOCTOR.search(answer)) if ex["kind"] == "safety" else None),
        "numeric_date": bool(NUMERIC_DATE.search(answer)),
    })

def rate(key):
    vals = [r[key] for r in rows if r[key] is not None]
    return f"{sum(vals)}/{len(vals)}" if vals else "n/a"

print("Cites when it should:", rate("cites_when_should"))
print("All citations valid:", rate("valid_citations"))
print("Abstains on unanswerable questions:", rate("abstains"))
print("Refers safety questions to a doctor:", rate("refers_to_doctor"))
print("Numeric dates (should be 0):", sum(r["numeric_date"] for r in rows))

In [ ]:
# 8. Read a few answers yourself (one of each kind)
seen = set()
for ex in val_raw:
    if ex["kind"] in seen:
        continue
    seen.add(ex["kind"])
    print(f"=== {ex['kind']}: {ex['messages'][1]['content']}")
    print(generate(ex["messages"][:2]), "\n")

In [ ]:
# 9. Push the merged model to a NEW repo (…-v2), so the model the app uses now is not replaced until the
# new one wins the benchmark (see training/README.md, "Phase 4").
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your Hugging Face access token (write access): ")
login(token=hf_token)
HF_USERNAME = input("Your Hugging Face username: ").strip()
REPO_ID = f"{HF_USERNAME}/neura-care-assistant-v2"
model.push_to_hub_merged(REPO_ID, tokenizer, save_method="merged_16bit", token=hf_token)
print(f"Done: https://huggingface.co/{REPO_ID}")